<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%EA%B0%90%EC%84%B1%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

### 지도학습 기반 감성 분석 실습

IMDB의 영화 사이트의 영화평 이용
(지도학습 기반 감성 분석은 텍스트 기반의 이진 분류 모델)

tsv 파일은 탭(\t)문자로 분리된 파일입니다.


In [ ]:
!unzip /content/labeledTrainData.tsv.zip -d .

Archive:  /content/labeledTrainData.tsv.zip
  inflating: ./labeledTrainData.tsv  


In [ ]:
review_df=pd.read_csv('./labeledTrainData.tsv',header=0,sep='\t',quoting=3)
review_df.head(3)

,id,sentiment,review
0,"""5814_8""",1,"""With all this stuff going down at the moment ..."
1,"""2381_9""",1,"""\""The Classic War of the Worlds\"" by Timothy ..."
2,"""7759_3""",0,"""The film starts with a manager (Nicholas Bell..."


In [ ]:
print(review_df['review'][0])

"With all this stuff going down at the moment with MJ i've started listening to his music, watching the odd documentary here and there, watched The Wiz and watched Moonwalker again. Maybe i just want to get a certain insight into this guy who i thought was really cool in the eighties just to maybe make up my mind whether he is guilty or innocent. Moonwalker is part biography, part feature film which i remember going to see at the cinema when it was originally released. Some of it has subtle messages about MJ's feeling towards the press and also the obvious message of drugs are bad m'kay.<br /><br />Visually impressive but of course this is all about Michael Jackson so unless you remotely like MJ in anyway then you are going to hate this and find it boring. Some may call MJ an egotist for consenting to the making of this movie BUT MJ and most of his fans would say that he made it for the fans which if true is really nice of him.<br /><br />The actual feature film bit when it finally sta

html 형식에서 추출해 `<br/>` 태그가 존재합니다.

In [ ]:
import re
# <br />를 공백으로 변환
review_df['review']=review_df['review'].str.replace('<br />',' ')
# 파이썬의 정규 표현식 모듈인 re를 이용해 영어 문자열이 아닌 문자는 모두 공백으로 변환
review_df['review']=review_df['review'].apply(lambda x:re.sub("[^a-zA-Z]"," ",x)
)

In [ ]:
from sklearn.model_selection import train_test_split

X_features=review_df['review']
y_target=review_df['sentiment']

X_train,X_test,y_train,y_test=train_test_split(X_features,y_target,test_size=0.3,random_state=156)

In [ ]:
X_train.shape,X_test.shape

((17500,), (7500,))

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,roc_auc_score

# 스톱워드는 English,ngram은 (1,2)로 설정해 CountVectorization 수행
pipeline=Pipeline([
    ('cnt_vect',CountVectorizer(stop_words='english',ngram_range=(1,2))),
    ('lr_clf',LogisticRegression(solver='liblinear',C=10))
])

pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test)
pred_probs=pipeline.predict_proba(X_test)[:,1]

print(f'예측 정확도는 {accuracy_score(y_test,pred):.3f},ROC-AUC는 {roc_auc_score(y_test,pred_probs):.3f}')

예측 정확도는 0.886,ROC-AUC는 0.950


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
# TF-IDF 벡터화를 적용해 다시 예측 성능 측정
pipeline=Pipeline([
    ('tfidf_vect',TfidfVectorizer(stop_words='english',ngram_range=(1,2))),
    ('lr_clf',LogisticRegression(solver='liblinear',C=10))
])
pipeline.fit(X_train,y_train)
pred=pipeline.predict(X_test)
pred_probs=pipeline.predict_proba(X_test)[:,1]

print(f'예측 정확도는 {accuracy_score(y_test,pred):.3f},ROC-AUC는 {roc_auc_score(y_test,pred_probs):.3f}')

예측 정확도는 0.894,ROC-AUC는 0.960


TF-IDF 기반 피처 벡터화의 예측 성능

### 비지도 학습 기반 감성 분석 소개

Lexicon을 기반으로 합니다.

#### SentiWordNet을 이용한 감성 분석


In [ ]:
import nltk
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

True

In [ ]:
# 'present' 단어에 대한 Synset을 추출하겠습니다.
from nltk.corpus import wordnet as wn

term='present'

# 'present'라는 단어로 wordnet의 synsets 생성
synsets=wn.synsets(term)
print(f'synsets() 반환 type:{type(synsets)}')
print(f'synsets() 반환 값 개수type:{len(synsets)}')
print(f'synsets() 반환 값:{synsets}')

synsets() 반환 type:<class 'list'>
synsets() 반환 값 개수type:18
synsets() 반환 값:[Synset('present.n.01'), Synset('present.n.02'), Synset('present.n.03'), Synset('show.v.01'), Synset('present.v.02'), Synset('stage.v.01'), Synset('present.v.04'), Synset('present.v.05'), Synset('award.v.01'), Synset('give.v.08'), Synset('deliver.v.01'), Synset('introduce.v.01'), Synset('portray.v.04'), Synset('confront.v.03'), Synset('present.v.12'), Synset('salute.v.06'), Synset('present.a.01'), Synset('present.a.02')]


Synset('present.n.01')와 같이 Synset 객체의 파라미터 'present.n.01'은 POS 태그를 나타냅니다.

present는 의미, n은 명사 품사,01은 present가 명사로서 가지는 의미가 여러 가지가 있어서 이를 구분하는 인덱스입니다.

Synset은 POS,정의(Definition),부명제(Lemma)등으로 시맨틱적인 요소를 표현할 수 있습니다.

In [ ]:
for synset in synsets:
  print(f'##### Synset name:{synset.name()} #####')
  print(f'POS:{synset.lexname()}')
  print(f'Definition:{synset.definition()}')
  print(f'Lemmas:{synset.lemma_names()}')

##### Synset name:present.n.01 #####
POS:noun.time
Definition:the period of time that is happening now; any continuous stretch of time including the moment of speech
Lemmas:['present', 'nowadays']
##### Synset name:present.n.02 #####
POS:noun.possession
Definition:something presented as a gift
Lemmas:['present']
##### Synset name:present.n.03 #####
POS:noun.communication
Definition:a verb tense that expresses actions or states at the time of speaking
Lemmas:['present', 'present_tense']
##### Synset name:show.v.01 #####
POS:verb.perception
Definition:give an exhibition of to an interested audience
Lemmas:['show', 'demo', 'exhibit', 'present', 'demonstrate']
##### Synset name:present.v.02 #####
POS:verb.communication
Definition:bring forward and present to the mind
Lemmas:['present', 'represent', 'lay_out']
##### Synset name:stage.v.01 #####
POS:verb.creation
Definition:perform (a play), especially on a stage
Lemmas:['stage', 'present', 'represent']
##### Synset name:present.v.04 #####
P

Synset('present.n.01')과 Synset('present'.n.02')는 명사지만 서로 다른 의미를 가지고 있습니다.

Synset('present.n.01')은 POS가 noun.time 이며 Definition을 살펴보면 '시간적인 의미로 현재'를 나타냅니다. Synset('present.n.02')는 POS가 noun.possession이며 Definition은 '선물' 입니다.

이처럼 synset은 하나의 단어가 가질 수 있는 여러 가지 시맨틱 정보를 개별 클래스로 나타냅니다.

In [ ]:
## 단어 유사도 ##
# path_similarity:sysnet() 단어 간의 유사도
# sysnet 객체를 단어별로 생성
tree=wn.synset('tree.n.01')
lion=wn.synset('lion.n.01')
tiger=wn.synset('tiger.n.02')

entities=[tree,lion,tiger]
similarities=[]
entity_names=[entity.name().split('.')[0]for entity in entities]

# 단어별 synset을 반복하면서 다른 단어의 synset과 유사도를 측정합니다.
for entity in entities:
  similarity=[round(entity.path_similarity(compared_entity),2)
  for compared_entity in entities]
  similarities.append(similarity)

# 개별 단어별 synset과 다른 단어의 synset과의 유사도를 DataFrame 형태로 지정
similarity_df=pd.DataFrame(similarities,columns=entity_names,index=entity_names)
similarity_df

,tree,lion,tiger
tree,1.00,0.07,0.07
lion,0.07,1.00,0.33
tiger,0.07,0.33,1.00


In [ ]:
import nltk
from nltk.corpus import sentiwordnet as swn

senti_synsets = list(swn.senti_synsets('slow'))
print(f'senti_synsets() 반환 type:{type(senti_synsets)}')
print(f'synsets() 반환 값 개수type:{len(senti_synsets)}')
print(f'synsets() 반환 값:{senti_synsets}')

senti_synsets() 반환 type:<class 'list'>
synsets() 반환 값 개수type:11
synsets() 반환 값:[SentiSynset('decelerate.v.01'), SentiSynset('slow.v.02'), SentiSynset('slow.v.03'), SentiSynset('slow.a.01'), SentiSynset('slow.a.02'), SentiSynset('dense.s.04'), SentiSynset('slow.a.04'), SentiSynset('boring.s.01'), SentiSynset('dull.s.08'), SentiSynset('slowly.r.01'), SentiSynset('behind.r.03')]


Sentisynset 객체는 단어의 감성을 나타내는 감성 지수와 객관성(감성과 반대)을 나타내는 객관성 지수를 가지고 있습니다.

어떤 단어가 전혀 감성적이지 않으면 객관성 지수는 1이 되고, 감성 지수는 모두 0이 됩니다.

In [ ]:
import nltk
from nltk.corpus import sentiwordnet as swn

father=swn.senti_synset('father.n.01')
print(f'father 긍정감성 지수:{father.pos_score()}')
print(f'father 부정감성 지수:{father.neg_score()}')
print(f'father 객관성 지수:{father.obj_score()}')
print('\n')
fabulous = swn.senti_synset('fabulous.a.01')
print(f'fabulous 긍정감성 지수:{fabulous.pos_score()}')
print(f'fabulous 부정감성 지수:{fabulous.neg_score()}')

father 긍정감성 지수:0.0
father 부정감성 지수:0.0
father 객관성 지수:1.0


fabulous 긍정감성 지수:0.875
fabulous 부정감성 지수:0.125


father은 객관적인 단어로 객관성 지수가 1.0이고 긍정 감성/부정 감성 지수 모두 0입니다.

반면에 fabulous는 감성 단어로서 긍정 감성 지수가 0.875, 부정 감성 지수가 0.125 입니다.

### SentiWordNet을 이용한 영화 감상평 감성 분석

1. 문서(Document)를 문장(Sentence)단위로 분해
2. 다시 문장을 단어(Word)단위로 토큰화하고 품사 태깅
3. 품사 태깅된 단어 기반으로 synset 객체와 senti_synset 객체를 생성
4. senti_synset 객체에서 긍정 감성/부정 감성 지수를 구하고 이를 모두 합산해 특정 임계치 값 이상일때 긍정 감성으로, 그렇지 않을 떄에는 부정 감성으로 결정


SentiWordNet을 이용하기 위해서 WordNet을 이용해 문서를 다시 단어로 토큰화한 뒤 어근 추출(Lemmatization)과 품사 태깅(POS Tagging)을 적용해야 합니다.

In [ ]:
from nltk.corpus import wordnet as wn

# 간단한 NTLK PennTreebank Tag를 기반으로 WordNet 기반의 품사 Tag로 변환
def penn_to_wn(tag):
  if tag.startswith('J'):
    return wn.ADJ

  elif tag.startswith('N'):
    return wn.NOUN
  elif tag.startswith('R'):
    return wn.ADV
  elif tag.startswith('V'):
    return wn.VERB

In [ ]:
### 긍정 및 부정 감성 예측 ###
# 문서를 문장 -> 단어토큰 -> 품사 태깅
from nltk.stem import WordNetLemmatizer
from nltk.corpus import sentiwordnet as swn
from nltk import sent_tokenize,word_tokenize,pos_tag

def swn_polarity(text):
  # 감성 지수 초기화
  sentiment=0.0
  tokens_count=0

  lemmatizer=WordNetLemmatizer()
  raw_sentences=sent_tokenize(text)
  # 분해된 문장별로 단어 토큰 -> 품사 태깅 후에 SentiSynset 생성 -> 감성 지수 합산
  for raw_sentence in raw_sentences:
    #NLTK 기반의 품사 태깅 문장 추출
    tagged_sentence=pos_tag(word_tokenize(raw_sentence))
    for word,tag in tagged_sentence:
      #WordNet 기반 품사 태깅과 어근 추출
      wn_tag=penn_to_wn(tag)
      if wn_tag not in (wn.NOUN,wn.ADJ,wn.ADV):
        continue
      lemma=lemmatizer.lemmatize(word,pos=wn_tag)
      if not lemma:
        continue
      # 어근을 추출한 단어와 WordNet 기반 품사 태깅을 입력해 Synset 객체 생성
      synsets=wn.synsets(lemma,pos=wn_tag)
      if not synsets:
        continue
      #sentiwordnet의 감성 단어 분석으로 synset 추출
      # 모든 단어에 대해 긍정 감성 지수는 +로 부정 감성 지수는 -로 합산해 감성 지수 계산
      synset = synsets[0]
      swn_synset=swn.senti_synset(synset.name())
      sentiment += (swn_synset.pos_score()- swn_synset.neg_score())
      tokens_count+=1

  if not tokens_count:
    return 0

  # 총 score가 0이상일 경우 긍정(Positive) 1,그렇지 않을 경우 부정(Negative)0 반환
  if sentiment >= 0:
    return 1

  return 0


In [ ]:
review_df.columns

Index(['id', 'sentiment', 'review'], dtype='object')

In [ ]:
review_df['preds']=review_df['review'].apply(lambda x:swn_polarity(x))
y_target=review_df['sentiment'].values
preds=review_df['preds'].values

In [ ]:
# SentiWordNet의 감성 분석 예측
from sklearn.metrics import accuracy_score,confusion_matrix,precision_score
from sklearn.metrics import recall_score,f1_score,roc_auc_score
print(confusion_matrix(y_target,preds))
print(f'정확도:{np.round(accuracy_score(y_target,preds),4)}')
print(f'정밀도:{np.round(precision_score(y_target,preds),4)}')
print(f'재현율:{np.round(recall_score(y_target,preds),4)}')

[[7669 4831]
 [3635 8865]]
정확도:0.6614
정밀도:0.6473
재현율:0.7092


정확도가 66.14%, 정밀도가 64.73% 입니다.

정확도 지표를 포함한 전반적인 성능 평가 지표는 만족스러울 만한 수치는 아닌 것 같습니다.

### VADER를 이용한 감성 분석

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

senti_analyzer=SentimentIntensityAnalyzer()
# polarity_scores():감성 점수가 특정 임계값 이상이면 긍정, 그렇지 않으면 부정 판단
# neg: 부정 감성 지수
# neu:중립적인 감정 지수
# pos: 긍정 감정 지수
# compound : -1~1 사이의 감성 지수
# compound score : 0.1 이상이면 긍정 감성, 그 이하면 부정 감성
senti_scores=senti_analyzer.polarity_scores(review_df['review'][0])
print(senti_scores)

{'neg': 0.13, 'neu': 0.743, 'pos': 0.127, 'compound': -0.7943}


VADER을 이용해 IMDB 감성 분석 수행

vader_polarity() 함수 생성, vader_polarity() 함수는 입력 파라미터로 영화 감상평 텍스트와 긍정/부정을 결정하는 임계값(threshold)을 가지고 감성 결과 반환

In [60]:
def vader_polarity(review,threshold=0.1):
  analyzer=SentimentIntensityAnalyzer()
  scores=analyzer.polarity_scores(review)

  #Compound 값에 기반해 threshold 입력값보다 크면 1, 그렇지 않으면 0 반환
  agg_score=scores['compound']
  final_sentiment = 1 if agg_score >= threshold else 0
  return final_sentiment

# apply lambda 식을 이용해 레코드 별로 vader_polarity()를 수행하고 결과를 'vader_preds'에 저장
review_df['vader_preds']=review_df['review'].apply(lambda x:vader_polarity(x,0.1))
y_target=review_df['sentiment'].values
vader_preds=review_df['vader_preds'].values

print(confusion_matrix(y_target,vader_preds))
print(f'정확도: {np.round(accuracy_score(y_target, vader_preds), 4)}')
print(f'정밀도: {np.round(precision_score(y_target, vader_preds), 4)}')
print(f'재현율: {np.round(recall_score(y_target, vader_preds), 4)}')

[[ 6747  5753]
 [ 1858 10642]]
정확도: 0.6956
정밀도: 0.6491
재현율: 0.8514


정확도가 SentiWordNet 보다 향상됐고, 특히 재현율은 약 85.14%로 매우 크게 향상됐습니다.

감성 사전을 이용한 감성 분석 예측 성능은 지도학습 분류 기반의 예측 성능에 비해 아직은 낮은 수준이지만 결정 클래스 값이 없는 상황을 고려한다면 예측 성능에 일정 수준 만족할 수 있을 것입니다.